# Exploratory Data Analysis — 30-Day Readmission

This notebook mirrors `scripts/run_eda.py`. Run `python scripts/run_pipeline.py` first
to download the dataset and generate the figures referenced below.

**Dataset:** UCI Diabetes 130-US Hospitals (1999–2008), CC BY 4.0.

In [ ]:
import pandas as pd

from src.data.download import download_dataset, load_dataset

pd.set_option("display.max_columns", 60)
frame = load_dataset(download_dataset())
frame.shape

## Target definition

`readmitted == "<30"` is the positive class (readmission within 30 days).

In [ ]:
target = (frame["readmitted"].astype(str).str.strip() == "<30").astype(int)
print(frame["readmitted"].value_counts())
print(f"Positive rate: {target.mean():.2%} ({target.sum():,} / {len(target):,})")
print(f"Unique patients: {frame['patient_nbr'].nunique():,}")

## Missingness

`weight` is almost entirely missing; `medical_specialty` and `payer_code` are
missing for roughly half of encounters. The pipeline drops `weight` and
`payer_code`, and keeps specialty as an explicit `Unknown` level.

In [ ]:
missing = frame.isna().mean().sort_values(ascending=False)
missing[missing > 0].head(10).to_frame("missing_share")

## Readmission by age and prior utilization

Prior inpatient utilization is the strongest exploratory signal.

In [ ]:
frame.assign(target=target).groupby("age", observed=True)["target"].mean().mul(100).round(2)

In [ ]:
binned = frame["number_inpatient"].clip(upper=3)
frame.assign(target=target, prior_inpatient=binned).groupby("prior_inpatient", observed=True)["target"].mean().mul(100).round(2)

## Generated figures

![Target distribution](../reports/figures/target_distribution.png)

![Readmission by prior inpatient visits](../reports/figures/readmission_by_prior_inpatient.png)

![ROC curves](../reports/figures/roc_curves.png)

See `reports/figures/` for the full set and `reports/final_report.md` for the analysis.